# USGS Earthquakes — full feature demo

Generates a self-contained offline HTML report from the USGS real-time earthquake feed.

Demonstrates every dfreport feature in one notebook:
- **Date range filter** (header bar) driving charts, stat panel, and data table
- **Categorical dropdowns** — `magType`, `type` auto-detected from low-cardinality columns
- **Numeric range inputs** — `mag`, `depth`
- **Text substring search** — `place`
- **Two scatter charts** — spatial (lon/lat) and magnitude vs depth
- **Per-day stat table** filtered by the date range

**Dataset:** USGS Earthquake Hazards Program real-time feed — M2.5+ events, last 30 days.  
URL: https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php  
License: Public domain (US federal government data, 17 U.S.C. § 105).

In [ ]:
# --- path setup (local development) ---
# Remove these two lines once dfreport is installed via pip.
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../..').resolve()))

import pandas as pd
from dfreport import Report, ScatterChart, TOTAL_ROW_FLAG

In [ ]:
# Load the USGS M2.5+ feed (last 30 days, updates every minute)
url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.csv'
raw_df = pd.read_csv(url)

print(f'Raw feed: {len(raw_df)} events, columns: {list(raw_df.columns)}')

In [ ]:
# Parse time → ISO date string (YYYY-MM-DD).
# The date filter in dfreport compares strings lexicographically, so ISO format is required.
raw_df['date'] = pd.to_datetime(raw_df['time']).dt.strftime('%Y-%m-%d')

# Keep only the columns we want to display
df = raw_df[['date', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'place', 'type']].copy()

# Drop rows with missing magnitude or depth (rare but possible near the feed boundary)
df = df.dropna(subset=['mag', 'depth']).reset_index(drop=True)

print(f'{len(df)} events from {df["date"].min()} to {df["date"].max()}')
df.head()

In [ ]:
# Build per-day stat table
daily_stats = (
    df.groupby('date', as_index=False)
    .agg(
        count=('mag', 'count'),
        max_mag=('mag', 'max'),
        mean_mag=('mag', 'mean'),
        mean_depth=('depth', 'mean'),
    )
    .sort_values('date', ascending=False)
    .round(2)
    .reset_index(drop=True)
)

# Prepend an ALL row (marked TOTAL_ROW_FLAG=True for bold styling)
total_row = pd.DataFrame([{
    'date':       'ALL',
    'count':      len(df),
    'max_mag':    round(df['mag'].max(), 2),
    'mean_mag':   round(df['mag'].mean(), 2),
    'mean_depth': round(df['depth'].mean(), 2),
    TOTAL_ROW_FLAG: True,
}])
daily_stats = pd.concat([total_row, daily_stats], ignore_index=True)

stat_col_defs = [
    {'key': 'date',       'label': 'Date',       'fmt': 'str'},
    {'key': 'count',      'label': 'Events',     'fmt': 'int'},
    {'key': 'max_mag',    'label': 'Max Mag',    'fmt': 'str'},
    {'key': 'mean_mag',   'label': 'Avg Mag',    'fmt': 'str'},
    {'key': 'mean_depth', 'label': 'Avg Depth',  'fmt': 'str'},
]

daily_stats.head()

In [ ]:
report_path = (
    Report(df, title='USGS Earthquakes — M2.5+, last 30 days')
    # Date range picker in the header bar — keys on the 'date' column
    .date_filter('date')
    # Categorical dropdowns: magType (ml, md, mw, …) and type (earthquake, quarry blast, …)
    .filters(['magType', 'type'])
    # Spatial chart: longitude on X, latitude on Y → map-like view
    .chart(
        ScatterChart(
            x='longitude',
            y='latitude',
            title='Earthquake Locations',
            x_label='Longitude',
            y_label='Latitude',
            equal_axes=True,   # lock 1:1 scale for a geographically faithful view
            hover_columns=['place', 'mag', 'depth', 'date', 'magType'],
        )
    )
    # Second chart: magnitude vs depth — reveals depth-clustering behaviour
    .chart(
        ScatterChart(
            x='mag',
            y='depth',
            title='Magnitude vs Depth',
            x_label='Magnitude',
            y_label='Depth (km)',
            equal_axes=False,
            hover_columns=['place', 'date', 'magType', 'type'],
        )
    )
    # .summary_panel('mag', 'depth', label='Mag / depth stats')
    .stat_table(daily_stats, stat_col_defs, title='Events per day')
    .table(
        categorical_threshold=20,  # magType has up to ~15 codes; raise threshold to catch them
        col_overrides={
            # latitude and longitude are numeric, which is correct — no overrides needed.
            # 'place' has high cardinality → auto-detected as text search, which is correct.
        },
        title='Event list',
    )
    .save('earthquakes_report.html')
)

print(f'Saved: {report_path.name}')

## What you should see

Open `earthquakes_report.html` in any browser.

### Header bar
- **Date range pickers** (Start / End) — narrows every chart, the stat panel, and the event table.  
  The per-day stat table also updates to show only days within the selected range.
- **Mag Type dropdown** — filter to a specific magnitude scale (ml, md, mw, …)
- **Type dropdown** — filter to earthquakes, quarry blasts, explosions, etc.

### Charts
- **Earthquake Locations** — lon/lat scatter with equal-axes lock.  
  The crosshair marks the average longitude and latitude of the visible events.
- **Magnitude vs Depth** — reveals shallow-focus vs deep-focus clustering.  
  Many subduction-zone earthquakes cluster at 50–150 km depth.

### Data table column types (auto-detected)
| Column | Detected as | Filter rendered |
|---|---|---|
| `date` | text (ISO string, high cardinality) | substring search |
| `latitude`, `longitude`, `depth`, `mag` | numeric (float) | min/max range |
| `magType`, `type` | categorical (≤20 unique values) | exact-match dropdown |
| `place` | text (high cardinality) | substring search |

### Reproducing with different magnitude thresholds

The USGS feed also provides:
- All magnitudes: `.../summary/all_month.csv` (~10 000+ events, large file)
- M4.5+: `.../summary/4.5_month.csv` (~400 events)
- M1.0+: `.../summary/1.0_month.csv` (~5 000 events)

Replace the URL in the load cell to change the threshold.